### Team Members:


In [54]:
# Importing our packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import statsmodels.api as sm

# Loading our movies dataset
movies = pd.read_csv("tmdb_5000_movies.csv")

# Introduction

The film industry is a high-risk, high-reward environment where production budgets can range from thousands to hundreds of millions of dollars. Despite the availability of historical data, predicting whether a movie will be financially successful remains a challenging problem. Studios must make investment decisions before release, often relying on incomplete or uncertain information.

In this project, we aim to use data-driven techniques to better understand what factors contribute to a movie’s financial success. By leveraging historical movie data, we can identify patterns and build predictive models that may assist in decision-making.

### Project Goal

The primary goal of this project is to classify whether a movie is a financial <strong>hit</strong> or  <strong>flop</strong> based on its characteristics. Rather than using raw revenue as a measure of success, we define success using Return on Investment (ROI): 
$$
ROI = \frac{Revenue - Budget}{Budget}
$$
Using ROI provides a more meaningful measure of financial performance, as it accounts for the scale of investment. 

Therefore, we will:
- Engineer relevant features from the dataset
- Define a threshold to classify movies as “hits” or “flops”
- Build and compare multiple classification models
- Use cross-validation and regularization to optimize model performance
- Interpret which factors most strongly influence success

### Data Description

The dataset used in this project is the TMDB 5000 Movies Dataset, which contains information on 4803 movies.

Key features include:

- Budget and Revenue
- Genres (categorical, multi-label)
- Runtime
- Popularity
- Vote average and vote count
- Production companies and countries

The dataset requires preprocessing, as several columns are stored as nested JSON strings and must be parsed and transformed into usable features.

In [55]:
# Dimensions of our dataset
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   str    
 2   homepage              1712 non-null   str    
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   str    
 5   original_language     4803 non-null   str    
 6   original_title        4803 non-null   str    
 7   overview              4800 non-null   str    
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   str    
 10  production_countries  4803 non-null   str    
 11  release_date          4802 non-null   str    
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   str    
 15  status                4803 non-n

### Data Cleaning and Preprocessing

In [56]:
movies.isnull().sum().sort_values(ascending=False)

homepage                3091
tagline                  844
overview                   3
runtime                    2
release_date               1
id                         0
budget                     0
genres                     0
original_title             0
popularity                 0
original_language          0
keywords                   0
production_countries       0
production_companies       0
spoken_languages           0
revenue                    0
status                     0
title                      0
vote_average               0
vote_count                 0
dtype: int64

In [57]:
# Drop irrelevant columns from our dataframe
columns_to_drop = ['homepage', 'id', 'overview', 'tagline', 'title', 'keywords', 'spoken_languages']
movies = movies.drop(columns=columns_to_drop)

In [58]:
# Remove movies with no budget and revenue since they are not useful
movies = movies[(movies['budget'] > 0) & (movies['revenue'] > 0)]

# Drop remaining missing values
movies = movies.dropna()

In [59]:
# Function to extract names from JSON-like strings using regex.
def extract_names(column_str: str) -> list[str]:
    if pd.isna(column_str):
        return []
    return re.findall(r'"name"\s*:\s*"([^"]+)"', column_str)

In [60]:
json_columns = ['genres', 'production_companies', 'production_countries']

for col in json_columns:
    movies[col] = movies[col].apply(extract_names)

In [61]:
# ROI calculation
movies['ROI'] = (movies['revenue'] - movies['budget']) / movies['budget']

In [62]:
# Define success based on the ROI using 1.5 based on information found online for films (1.5 is slightly higher than the median in our data)
movies['success'] = (movies['ROI'] > 1.5).astype(int)

In [63]:
movies.head(5)

,budget,genres,original_language,original_title,popularity,production_companies,production_countries,release_date,revenue,runtime,status,vote_average,vote_count,ROI,success
0,237000000,"[Action, Adventure, Fantasy, Science Fiction]",en,Avatar,150.437577,"[Ingenious Film Partners, Twentieth Century Fo...","[United States of America, United Kingdom]",2009-12-10,2787965087,162.0,Released,7.2,11800,10.763566,1
1,300000000,"[Adventure, Fantasy, Action]",en,Pirates of the Caribbean: At World's End,139.082615,"[Walt Disney Pictures, Jerry Bruckheimer Films...",[United States of America],2007-05-19,961000000,169.0,Released,6.9,4500,2.203333,1
2,245000000,"[Action, Adventure, Crime]",en,Spectre,107.376788,"[Columbia Pictures, Danjaq, B24]","[United Kingdom, United States of America]",2015-10-26,880674609,148.0,Released,6.3,4466,2.594590,1
3,250000000,"[Action, Crime, Drama, Thriller]",en,The Dark Knight Rises,112.312950,"[Legendary Pictures, Warner Bros., DC Entertai...",[United States of America],2012-07-16,1084939099,165.0,Released,7.6,9106,3.339756,1
4,260000000,"[Action, Adventure, Science Fiction]",en,John Carter,43.926995,[Walt Disney Pictures],[United States of America],2012-03-07,284139100,132.0,Released,6.1,2124,0.092843,0


# Exploratory Data Analysis

# Conclusion